# Playing with Sentence Embeddings

> **This notebook continues the word-embeddings notebook** (`notebook.ipynb`). There we
> gave *each word* a context-aware vector. Sections here pick up at **9** and ask the next
> natural question: how do we get **one vector for a whole sentence** — so we can compare
> *sentences* by meaning?

This is the machinery behind **semantic search** and **RAG**: embed every document as a
vector, embed the query, and return the nearest ones. By the end you'll have built a tiny
search engine that matches by *meaning*, not keywords.

In [1]:
# One-time install (skip if you already have these)
# %pip install transformers torch

## 9. From many word vectors to one sentence vector

In the last notebook, one forward pass gave us **one vector per token**. To describe a
*whole sentence* with a single vector, we have to **pool** those token vectors into one.
The simplest and most reliable recipe is **mean pooling** — just **average all the token
vectors** in the sentence.

Notice this is the exact same forward pass as `embed_word` — we just combine *all* the
token vectors instead of plucking one out.

In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel, logging

logging.set_verbosity_error()

# Start with plain, un-fine-tuned DistilBERT (same model as the word notebook)
tok_bert = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased").eval()
print("loaded distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

loaded distilbert-base-uncased


In [3]:
def embed_sentence(sentence, tokenizer, model):
    """Turn one sentence into a single vector by averaging its word vectors."""
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)
    token_vectors = out.last_hidden_state[0]   # (num_tokens, dim): one row per token
    return token_vectors.mean(0).numpy()       # average the rows -> one vector


def cosine(u, v):
    return float(u.dot(v) / (np.linalg.norm(u) * np.linalg.norm(v)))


vec = embed_sentence("A man is playing a guitar.", tok_bert, bert)
print("one sentence -> one vector, shape:", vec.shape)

one sentence -> one vector, shape: (768,)


## 10. Does plain BERT give *good* sentence vectors? (not really)

Now the honest test. We have four sentence pairs — two **related** (paraphrases) and two
**unrelated**. If the geometry is any good, related pairs should score clearly higher than
unrelated ones. Let's see.

In [4]:
pairs = [
    ("related",   "A man is playing a guitar.",     "Someone is playing a musical instrument."),
    ("unrelated", "A man is playing a guitar.",     "A dog is running through a field."),
    ("related",   "The stock market crashed today.", "Share prices fell sharply."),
    ("unrelated", "The stock market crashed today.", "She baked a chocolate cake."),
]

def show_pair_scores(tokenizer, model, title):
    print(title)
    rel, unrel = [], []
    for kind, a, b in pairs:
        va = embed_sentence(a, tokenizer, model)
        vb = embed_sentence(b, tokenizer, model)
        s = cosine(va, vb)
        (rel if kind == "related" else unrel).append(s)
        print(f"  [{kind:<9}] {s:.3f}   {a[:30]:<32} | {b}")
    print(f"  --> avg related {np.mean(rel):.3f}   vs   avg unrelated {np.mean(unrel):.3f}"
          f"   (gap {np.mean(rel) - np.mean(unrel):.3f})\n")

show_pair_scores(tok_bert, bert, "PLAIN DistilBERT (mean-pooled):")

PLAIN DistilBERT (mean-pooled):
  [related  ] 0.941   A man is playing a guitar.       | Someone is playing a musical instrument.
  [unrelated] 0.868   A man is playing a guitar.       | A dog is running through a field.
  [related  ] 0.870   The stock market crashed today   | Share prices fell sharply.


  [unrelated] 0.687   The stock market crashed today   | She baked a chocolate cake.
  --> avg related 0.906   vs   avg unrelated 0.778   (gap 0.128)



Look at the scale: **everything scores high**, even the unrelated pairs. Related vs.
unrelated are barely separated — a tiny gap. Plain BERT packs all sentences into a narrow
cone of the space (this is called **anisotropy**), so cosine can't tell meaning apart well.

Why? Think back to what BERT was trained on: **masked language modeling** — predict a
missing *word*. It was **never** asked to make *whole-sentence* vectors comparable. Good
sentence geometry was never part of its objective, so we only get it by accident. To fix
that, we need a model that was trained *for exactly this*.

## 11. SBERT: the same machinery, fine-tuned to fix the geometry

**Sentence-BERT (SBERT)** is a BERT that was given a *second* round of training aimed
squarely at sentence similarity. We'll load a popular one, **`all-MiniLM-L6-v2`** — small
and fast — and embed with the **exact same `embed_sentence` function**. The *only* thing
that changed is the weights.

In [5]:
tok_sbert = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
sbert = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").eval()

show_pair_scores(tok_sbert, sbert, "SBERT (all-MiniLM-L6-v2, mean-pooled):")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT (all-MiniLM-L6-v2, mean-pooled):
  [related  ] 0.697   A man is playing a guitar.       | Someone is playing a musical instrument.
  [unrelated] 0.103   A man is playing a guitar.       | A dog is running through a field.
  [related  ] 0.642   The stock market crashed today   | Share prices fell sharply.
  [unrelated] -0.003   The stock market crashed today   | She baked a chocolate cake.
  --> avg related 0.669   vs   avg unrelated 0.050   (gap 0.620)



Now the numbers **spread out**: related pairs stay high, unrelated pairs drop toward 0.
The gap is large and the ordering is trustworthy. Same pooling, same forward pass — the
fine-tuned weights are what made cosine similarity meaningful for sentences.

## 12. How was SBERT trained?

Plain BERT learned language from masked-word prediction but never optimized *sentence*
geometry. SBERT adds a fine-tuning stage that optimizes it **directly**, using **pairs of
sentences** run through a **siamese network** — the *same* encoder applied to both
sentences, so their vectors end up comparable.

The dominant recipe (used by `all-MiniLM-L6-v2`) is **contrastive learning on positive
pairs**, trained on **over a billion** naturally-occurring pairs — duplicate questions,
question↔answer, title↔body, paraphrases, translations. The loss (**Multiple Negatives
Ranking / InfoNCE**) works per batch:

- each sentence's true partner is its **positive** → pull their vectors **together**,
- every *other* sentence in the batch is a **negative** ("in-batch negatives") → push them
  **apart**.

The clever economics: positives occur naturally (cheap), and negatives are free because
you reuse the rest of the batch. That's what makes a billion-pair training run feasible.

> **The through-line of the whole chapter:** it's *still* "learn by prediction / contrast,"
> the unit just keeps growing. word2vec contrasted a **word** against its neighbors, MLM
> predicted a **masked word**, and SBERT contrasts a **whole sentence** against other
> sentences — so the cosine geometry we want for search is exactly what it was trained to
> get right.

## 13. The payoff: a tiny semantic search engine

This is what sentence embeddings are *for*. Embed a small collection of documents once,
embed a query, and rank documents by **cosine similarity** to the query. No keyword
overlap required — matches are by **meaning**. This is the retrieval core of **RAG**.

In [6]:
corpus = [
    "The mitochondria is the powerhouse of the cell.",
    "You can reset your password from the account settings page.",
    "Photosynthesis converts sunlight into chemical energy in plants.",
    "Our refund policy allows returns within 30 days of purchase.",
    "The Eiffel Tower is located in Paris, France.",
    "To change your login credentials, visit the security tab.",
    "Interest rates were raised by the central bank this quarter.",
]

# embed the whole corpus once (with SBERT), one sentence at a time
corpus_vecs = [embed_sentence(doc, tok_sbert, sbert) for doc in corpus]

def search(query, k=3):
    q = embed_sentence(query, tok_sbert, sbert)
    scored = sorted(((cosine(q, d), doc) for d, doc in zip(corpus_vecs, corpus)),
                    reverse=True)
    print(f"query: {query!r}")
    for score, doc in scored[:k]:
        print(f"   {score:.3f}   {doc}")
    print()

search("How do I recover my forgotten login?")
search("Can I get my money back?")
search("How do plants make food?")

query: 'How do I recover my forgotten login?'
   0.545   You can reset your password from the account settings page.
   0.390   To change your login credentials, visit the security tab.
   0.098   Our refund policy allows returns within 30 days of purchase.

query: 'Can I get my money back?'
   0.462   Our refund policy allows returns within 30 days of purchase.
   0.263   You can reset your password from the account settings page.
   0.174   Interest rates were raised by the central bank this quarter.

query: 'How do plants make food?'
   0.511   Photosynthesis converts sunlight into chemical energy in plants.
   0.275   The mitochondria is the powerhouse of the cell.
   -0.003   Interest rates were raised by the central bank this quarter.



Every top hit is right — and notice there's often **no shared keyword** ("recover login"
matched "reset your password"; "make food" matched "photosynthesis"). The match is on
*meaning*, carried entirely by the geometry of the embeddings. Swap this into a real
system and it's exactly how retrieval works in RAG.

## 14. Recap & where this goes next

| | word vectors (last notebook) | sentence vectors (this one) |
|---|---|---|
| one vector describes | one word in context | a whole sentence |
| how we get it | pluck one token's hidden state | **pool** all token vectors (mean) |
| plain BERT quality | great | **mediocre** (anisotropic) |
| the fix | — | **SBERT**: fine-tune on sentence *pairs* |
| training signal | masked-word prediction | contrastive positive/negative **pairs** |
| what it unlocks | word sense in context | **semantic search / RAG** |

You've now climbed the full ladder: **characters → words → words-in-context → sentences**,
and at every rung the same idea held — *train a model to predict or contrast, and useful
vectors fall out where geometry encodes meaning.*

We've been treating the encoder as a **black box** the whole way. The next chapter opens
it: we build **attention** from scratch — the mechanism that lets a word look at the other
words in its sentence — and see exactly how these context-aware vectors get computed.